In [45]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import cosine

def prepare_vectors(current_day, past_day):
    """
    Prepare numerical vectors for cosine similarity calculation.
    
    Parameters:
        current_day (pd.Series): The current day's metrics.
        past_day (pd.Series): The past day's metrics.
        
    Returns:
        Tuple: Two numpy arrays representing the vectors.
    """
    numeric_cols = current_day.index[current_day.apply(np.isreal)]
    v1 = np.nan_to_num(current_day[numeric_cols].astype(float).values)
    v2 = np.nan_to_num(past_day[numeric_cols].astype(float).values)
    return v1, v2

def find_reference_days_cosine(df, lookback=60, top_n=1):
    """
    Calculate cosine similarity between daily metrics for each day and past days.
    
    Parameters:
        df (pd.DataFrame): DataFrame containing daily metrics with a 'date' column.
        lookback (int): Number of past days to consider for similarity calculation.
        top_n (int): Number of closest reference days to save.
        
    Returns:
        pd.DataFrame: DataFrame with columns ['date', 'closest_reference_days', 'distance'].
    """
    results = []

    df = df.sort_values(by='date').reset_index(drop=True)

    for i in range(lookback, len(df)):
        current_day = df.iloc[i]
        past_days = df.iloc[i - lookback:i]

        distances = []
        for _, past_day in past_days.iterrows():
            v1, v2 = prepare_vectors(current_day.iloc[1:], past_day.iloc[1:])
            try:
                if np.all(v1 == 0) or np.all(v2 == 0):
                    dist = 1.0  
                else:
                    dist = cosine(v1, v2)

                distances.append((past_day['date'], dist))
            except Exception as e:
                print(f"Error calculating cosine distance: {e}")
                continue

        if distances:
            distances.sort(key=lambda x: x[1])
            
            closest_days = [d[0] for d in distances[:top_n]]
            closest_distance = distances[0][1] if distances else None

            results.append({
                'date': current_day['date'],
                'closest_reference_days': closest_days,
                'distance': closest_distance
            })

    return pd.DataFrame(results)

if __name__ == "__main__":
    input_file = '../src/data/processed/daily_metrics.csv'
    output_file = '../src/data/processed/reference_days_cosine_raw.csv'

    df = pd.read_csv(input_file)

    if 'Unnamed: 0' in df.columns:
        df.rename(columns={'Unnamed: 0': 'date'}, inplace=True)

    df['date'] = pd.to_datetime(df['date'])

    ref_cosine_df = find_reference_days_cosine(df, lookback=60, top_n=1)
    
    ref_cosine_df.to_csv(output_file, index=False)
    
    print(f"Cosine similarity analysis complete. Results saved to {output_file}")


Cosine similarity analysis complete. Results saved to ../src/data/processed/reference_days_cosine_raw.csv


In [12]:
import pandas as pd

file_path = '../src/data/processed/reference_days_cosine_table.csv'
df = pd.read_csv(file_path)

print(df.head())
print(df.info())


         date                             closest_reference_days  distance
0  2024-08-01  [Timestamp('2024-06-25 00:00:00'), Timestamp('...  0.000318
1  2024-08-02  [Timestamp('2024-06-21 00:00:00'), Timestamp('...  0.002781
2  2024-08-03  [Timestamp('2024-07-18 00:00:00'), Timestamp('...  0.001405
3  2024-08-04  [Timestamp('2024-06-14 00:00:00'), Timestamp('...  0.003667
4  2024-08-05  [Timestamp('2024-07-10 00:00:00'), Timestamp('...  0.002323
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91 entries, 0 to 90
Data columns (total 3 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   date                    91 non-null     object 
 1   closest_reference_days  91 non-null     object 
 2   distance                91 non-null     float64
dtypes: float64(1), object(2)
memory usage: 2.3+ KB
None


In [36]:
import pandas as pd
import importlib
importlib.reload(pd)

df = pd.read_csv('../src/data/processed/reference_days_cosine_raw.csv')
print(df.columns)


Index(['date', 'closest_reference_days', 'distances'], dtype='object')


In [37]:
df.head()

,date,closest_reference_days,distances
0,2024-03-01,"[datetime.date(2024, 2, 16), datetime.date(202...","[0.0006005654057744669, 0.0008472571465449885,..."
1,2024-03-02,"[datetime.date(2024, 3, 1), datetime.date(2024...","[0.0005215576886349327, 0.0007037757169819914,..."
2,2024-03-03,"[datetime.date(2024, 2, 25), datetime.date(202...","[0.0011880151799577021, 0.0013146561815523539,..."
3,2024-03-04,"[datetime.date(2024, 1, 31), datetime.date(202...","[0.0008117715588242813, 0.0011596223758825186,..."
4,2024-03-05,"[datetime.date(2024, 2, 26), datetime.date(202...","[0.0010681148516507033, 0.001146211206179526, ..."


In [18]:
import pandas as pd

input_file = '../src/data/preprocessed/meteo_da_price_cleaned.csv'
df = pd.read_csv(input_file)


print(df.head())
print(df.info())


  Timestamp start (Europe/Brussels) Timestamp end (Europe/Brussels)  \
0                  01/01/2024 00:00                01/01/2024 00:15   
1                  01/01/2024 00:15                01/01/2024 00:30   
2                  01/01/2024 00:30                01/01/2024 00:45   
3                  01/01/2024 00:45                01/01/2024 01:00   
4                  01/01/2024 01:00                01/01/2024 01:15   

   Price average forecast ECMWF ENS United Kingdom day-ahead (£/MWh)  \
0                                              45.38                   
1                                              45.38                   
2                                              45.38                   
3                                              45.38                   
4                                              43.90                   

                        dtbe                      dtutc  
0  2024-01-01 00:00:00+01:00  2023-12-31 23:00:00+00:00  
1  2024-01-01 00:15:00+0

In [21]:
print(daily_vectors.head())  
print(f"Number of daily vectors: {len(daily_vectors)}")

Series([], Name: Price average forecast ECMWF ENS United Kingdom day-ahead (£/MWh), dtype: object)
Number of daily vectors: 0


In [25]:
print(f"Results: {results[:5]}")  


Results: []


In [42]:
import pandas as pd

file_path = '../src/data/processed/daily_metrics.csv'
df = pd.read_csv(file_path)

print("Columns:", df.columns)
print(df.head())


Columns: Index(['Unnamed: 0', 'demand_National Demand Forecast (NDF) - GB (MW)_mean',
       'demand_National Demand Forecast (NDF) - GB (MW)_max',
       'demand_National Demand Forecast (NDF) - GB (MW)_min',
       'demand_National Demand Forecast (NDF) - GB (MW)_spread',
       'demand_National Demand Forecast (NDF) - GB (MW)_std',
       'price_Price average forecast ECMWF ENS United Kingdom day-ahead (£/MWh)_mean',
       'price_Price average forecast ECMWF ENS United Kingdom day-ahead (£/MWh)_max',
       'price_Price average forecast ECMWF ENS United Kingdom day-ahead (£/MWh)_min',
       'price_Price average forecast ECMWF ENS United Kingdom day-ahead (£/MWh)_spread',
       'price_Price average forecast ECMWF ENS United Kingdom day-ahead (£/MWh)_std',
       'price_Price average forecast ECMWF ENS United Kingdom day-ahead (£/MWh)_rolling_std_7d',
       'price_Price average forecast ECMWF ENS United Kingdom day-ahead (£/MWh)_price_range',
       'price_Price average forecast E

In [43]:
print("Sample of daily metrics:")
print(df.head())

print("Columns:", df.columns)


Sample of daily metrics:
   Unnamed: 0  demand_National Demand Forecast (NDF) - GB (MW)_mean  \
0  2024-05-31                                       19731.500000      
1  2024-06-01                                       20013.333333      
2  2024-06-02                                       18956.500000      
3  2024-06-03                                       23517.583333      
4  2024-06-04                                       23226.458333      

   demand_National Demand Forecast (NDF) - GB (MW)_max  \
0                                            20098.0     
1                                            24200.0     
2                                            24718.5     
3                                            27674.5     
4                                            26778.0     

   demand_National Demand Forecast (NDF) - GB (MW)_min  \
0                                            19365.0     
1                                            17441.0     
2                        